# 03 · Train with Training Hub: SFT, then GRPO

**What you will do**
- **Part A · SFT.** Teach Granite 4.1-3B the Edge Copilot job (radio note → exact JSON report) with `lora_sft`.
  First in this notebook, then the same call as a **TrainJob** on cluster GPUs.
- **Part B · GRPO.** Teach the burst line with `lora_grpo`, using the rule checker from notebook 01 as the reward.

**GPU:** one GPU for each run (about 10 GB for SFT; 24 GB or more for GRPO).
**Time:** SFT 5–20 minutes depending on the recipe; GRPO depends on the GPU (we used a dedicated 48 GB GPU).

| | Dev environment | On OpenShift AI |
|---|---|---|
| Stack | Pin torch, transformers, peft, unsloth yourself | Red Hat-built training image, supported |
| Laptop → cluster | Rewrite the script | Same Training Hub call, handed to Kubeflow Trainer |
| GPU access | One person per box | GPU size from a menu; a fair queue across teams |
| Tracking | Your own MLflow | The project's MLflow, loss logged live |

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

## Part A · SFT for the Edge Copilot

### A1 · The recipe: one Training Hub call

One recipe, as a plain Python dictionary. The same dictionary is used in the notebook, as a TrainJob, or in a pipeline.
`QUICK = True` trains on 1,200 notes for one epoch (about 5 minutes) to try the flow end to end; set it to `False`
for the full run (2,400 notes, 2 epochs, about 16 minutes on a ~10 GB GPU share).

In [ ]:
from training_hub import lora_sft

params = {
    "model_path": "/workspace/models/granite-4.1-3b",
    "data_path": "/workspace/data/edgecopilot/train.jsonl",
    "ckpt_output_dir": "/workspace/checkpoints/edgecopilot-sft",
    "data_output_path": "/workspace/checkpoints/edgecopilot-sft/_data",
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.0,   # about 31 M trainable weights, a 62 MB adapter
    "num_epochs": 2, "learning_rate": 2e-4, "lr_scheduler": "cosine",
    "micro_batch_size": 2, "max_seq_len": 1024,
    "dataset_type": "chat_template", "field_messages": "messages",
    "warmup_steps": 10, "gradient_accumulation_steps": 1, "seed": 42, "load_in_4bit": False,
    "logging_steps": 5, "save_final_checkpoint": True, "nproc_per_node": 1, "nnodes": 1,
}

QUICK = True
if QUICK:   # a 1,200-note subset, one epoch
    lines = (WS / "data/edgecopilot/train.jsonl").read_text().splitlines()[:1200]
    (WS / "data/edgecopilot/train-quick.jsonl").write_text("\n".join(lines) + "\n")
    params.update(num_epochs=1, data_path="/workspace/data/edgecopilot/train-quick.jsonl")

# The challenger: same recipe, plus 600 radio-style notes (the style the first model handled worst)
params_v2 = {**params, "data_path": "/workspace/data/edgecopilot/train-v2.jsonl",
             "ckpt_output_dir": "/workspace/checkpoints/edgecopilot-sft-v2",
             "data_output_path": "/workspace/checkpoints/edgecopilot-sft-v2/_data"}

### A2 · Option 1: run it right here

If this workbench has a GPU, Training Hub runs in the notebook. `RUN_IN_WORKBENCH = True` trains every SFT run in
this notebook (A2, the challenger in A3, and B1); `False` hands each run to the cluster as a TrainJob, which needs
the `training-edit` role for this workbench (granted once by a cluster admin).

In [ ]:
RUN_IN_WORKBENCH = False      # True: train every SFT run in this notebook instead of as TrainJobs (needs a GPU)

def here(p):
    """The same recipe, with /workspace paths pointing at this workbench's mount of the volume."""
    return {k: v.replace("/workspace", str(WS)) if isinstance(v, str) else v for k, v in p.items()}

if RUN_IN_WORKBENCH:
    lora_sft(**here(params))        # option 1: in the workbench, if it has a GPU

### A3 · Option 2: the same call as a TrainJob on cluster GPUs

Kubeflow Trainer runs the call in pods built from the `training-hub` runtime, which the platform team maintains.
The job waits in the queue until a GPU is free, and its loss appears live in MLflow and in the dashboard
(Develop & train > Jobs).

In [ ]:
# Kubeflow Trainer: hand a Training Hub call to the cluster as a TrainJob
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer
from kubeflow.trainer.options.kubernetes import (ContainerOverride, PodSpecOverride,
                                                 PodTemplateOverride, PodTemplateOverrides)

try:
    client = TrainerClient()                          # the workbench's own identity
    runtime = next(r for r in client.list_runtimes() if r.name == "training-hub")   # maintained by the platform team
    print("Kubeflow Trainer ready, runtime:", runtime.name)
except Exception as e:
    runtime = None
    workbench = os.environ.get("HOSTNAME", "<workbench-name>").rsplit("-", 1)[0]
    print(f"This workbench cannot start TrainJobs yet ({type(e).__name__}).")
    print("Ask a cluster admin to grant it once:")
    print(f"  oc create rolebinding training-edit --clusterrole=training-edit "
          f"--serviceaccount={NAMESPACE}:{workbench} -n {NAMESPACE}")
    print("Meanwhile, set RUN_IN_WORKBENCH = True in A2 to train in this notebook (needs a GPU workbench).")

def submit(algorithm, params, gpus=1):
    """Run one Training Hub call on cluster GPUs. The project volume is mounted at /workspace."""
    if runtime is None:
        raise RuntimeError("TrainJobs are not available to this workbench yet: see the message printed above.")
    env = {k: v for k, v in os.environ.items() if k.startswith("MLFLOW_")}   # log to this project's MLflow
    env.update({"MLFLOW_TRACKING_AUTH": "kubernetes-namespaced", "MLFLOW_K8S_INTEGRATION": "true",
                "HF_HOME": "/workspace/.cache/huggingface"})
    return client.train(
        runtime=runtime,
        trainer=TrainingHubTrainer(algorithm=algorithm, func_args=params, env=env,
                                   resources_per_node={"cpu": 4, "memory": "24Gi", "nvidia.com/gpu": gpus}),
        options=[PodTemplateOverrides(PodTemplateOverride(target_jobs=["node"], spec=PodSpecOverride(
            node_selector=NODE_SELECTOR or None,
            volumes=[{"name": "work", "persistentVolumeClaim": {"claimName": PVC}},
                     {"name": "dshm", "emptyDir": {"medium": "Memory"}}],
            containers=[ContainerOverride(name="node", volume_mounts=[{"name": "work", "mountPath": "/workspace"},
                                                                      {"name": "dshm", "mountPath": "/dev/shm"}])])))])

def follow(job_name):
    for line in client.get_job_logs(job_name, follow=True):
        print(line, flush=True)

In [ ]:
if RUN_IN_WORKBENCH:
    print("Already trained in this notebook (A2); no TrainJob needed.")
else:
    job = submit(TrainingHubAlgorithms.LORA_SFT, params)
    print("TrainJob:", job)
    follow(job)

In [ ]:
# Optional: the challenger, with 600 extra radio-style notes. Notebook 06 compares the two.
if RUN_IN_WORKBENCH:
    lora_sft(**here(params_v2))
else:
    job_v2 = submit(TrainingHubAlgorithms.LORA_SFT, params_v2)
    follow(job_v2)

### A4 · Before vs after, on held-out notes

A quick check with the adapter loaded directly (needs a GPU in this workbench). Notebook 04 does the same through
the served endpoint, with no GPU here.

In [ ]:
import sys; sys.path.insert(0, str(WS / "code"))     # report_check.py and burst_rules.py, written by notebook 01
from report_check import check, scorecard as report_scorecard
from burst_rules import check as burst_check, scorecard as burst_scorecard

In [ ]:
CHECK_HERE = False

if CHECK_HERE:
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    base = str(WS / "models/granite-4.1-3b")
    tok = AutoTokenizer.from_pretrained(base); tok.padding_side = "left"
    test = [json.loads(l) for l in (WS / "data/edgecopilot/test.jsonl").open()][:20]
    prompts = [tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True) for r in test]
    golds = [json.loads(r["messages"][2]["content"]) for r in test]

    @torch.no_grad()
    def answers(model):
        b = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**b, max_new_tokens=320, do_sample=False, pad_token_id=tok.eos_token_id)
        return [tok.decode(o[b["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]

    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16).cuda().eval()
    print("before:", report_scorecard(answers(model), golds))
    model = PeftModel.from_pretrained(model, str(WS / "checkpoints/edgecopilot-sft")).eval()
    print("after :", report_scorecard(answers(model), golds))

**What we measured** (300 held-out notes, 94-token prompt): untuned 0% exact (6.3% even with a 1,449-token prompt),
`edgecopilot-sft` (2 epochs, 2,400 notes, 16 minutes on a ~10 GB GPU share) **92.7%**, `edgecopilot-sft-v2`
**99.7–100%**. The quick recipe reached 75% on 20 held-out notes in about 6 minutes.

## Part B · GRPO for the burst line

GRPO samples several answers per report, scores each with the checker, and moves the model toward the better ones.
It works best when the model is already *sometimes* right, so we start from SFT on the checker-approved teacher
answers from notebook 01.

### B1 · SFT on the 173 checker-approved teacher answers

In [ ]:
burst_sft = {**params, "data_path": "/workspace/data/burst/sft-filtered.jsonl",
             "ckpt_output_dir": "/workspace/checkpoints/burst-sft",
             "data_output_path": "/workspace/checkpoints/burst-sft/_data",
             "num_epochs": 3, "micro_batch_size": 4, "warmup_steps": 5}
if RUN_IN_WORKBENCH:
    lora_sft(**here(burst_sft))
else:
    job = submit(TrainingHubAlgorithms.LORA_SFT, burst_sft)
    follow(job)

### B2 · GRPO with the rule checker as reward

`lora_grpo` needs a Python reward function, so this step runs as a Kubernetes Job with a short script. The script is
written to the project volume by the next cell, so you can read every line. It runs three stages on one GPU:
merge the SFT adapter into the base model, train with GRPO, then pick the best checkpoint on validation and score it
once on the test set.

In [ ]:
%%writefile /opt/app-root/src/workspace/code/grpo_burst.py
"""GRPO for the burst line: merge SFT -> Training Hub lora_grpo -> best checkpoint on val -> score on test."""
import glob, json, os, shutil, sys
from pathlib import Path
sys.path.insert(0, "/workspace/code")

BASE, SFT = "/workspace/models/granite-4.1-3b", "/workspace/checkpoints/burst-sft"
MERGED, OUT = "/workspace/models/burst-sft-merged", "/workspace/checkpoints/burst-grpo"
ITERS = int(os.environ.get("ITERS", 24))

# --- Settings for the OpenShift AI 3.5 training image (Training Hub 0.9.2 with the ART backend) --------------------
import art.dev, asyncio
import training_hub.algorithms.lora_grpo as th_grpo
_eng, _init = art.dev.EngineArgs, art.dev.InitArgs
art.dev.EngineArgs = lambda **kw: _eng(**kw, max_model_len=2048, max_num_seqs=16)   # short answers: size for 2K tokens
art.dev.InitArgs = lambda **kw: _init(**kw, max_seq_length=2048)
def _in_process(self, params):            # run training in this process, so errors show in the Job log
    import unsloth  # noqa: F401
    th_grpo._install_megatron_bridge_stub()
    import art
    from art.local.backend import LocalBackend
    return asyncio.run(self._run_training(params, art, LocalBackend))
th_grpo.ARTLoRAGRPOBackend.execute_training = _in_process
import art.transformers.patches as _ap
from transformers import masking_utils
def _mask_args(config, inputs_embeds, attention_mask, past_key_values, position_ids, layer_idx, *rest):
    if position_ids is not None and position_ids.dim() == 3: position_ids = position_ids[0]
    return _ap._preprocess_mask_arguments(config, inputs_embeds, attention_mask, past_key_values, position_ids, layer_idx, *rest)
masking_utils._preprocess_mask_arguments = _mask_args
# --------------------------------------------------------------------------------------------------------------------

def merge():
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    if Path(MERGED, "config.json").exists(): return
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16)
    PeftModel.from_pretrained(m, SFT).merge_and_unload().save_pretrained(MERGED)
    for f in Path(BASE).iterdir():                       # same tokenizer and config files as the base model
        if f.is_file() and not f.name.endswith(".safetensors") and f.name != "model.safetensors.index.json":
            shutil.copy(f, MERGED)

def train():
    from training_hub import lora_grpo
    from burst_rules import reward_fn
    lora_grpo(backend="art", model_path=MERGED, ckpt_output_dir=OUT, data_path="/workspace/data/burst/train.jsonl",
              reward_fn=reward_fn,                       # the rule checker from notebook 01
              num_iterations=ITERS, group_size=8, prompt_batch_size=16, learning_rate=1e-5,
              temperature=0.1,                           # = 1.0 for this adapter: LoRA training skipped Granite's x10 logits scaling
              max_tokens=200, concurrency=16, lora_r=16, lora_alpha=32,
              gpu_memory_utilization=float(os.environ.get("GPU_UTIL", 0.45)), enforce_eager=True)

def evaluate():
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    from burst_rules import scorecard
    root = sorted(glob.glob(f"{OUT}/.art/*/models/*/checkpoints"))[0]
    steps = [f"{i:04d}" for i in range(4, ITERS + 1, 4) if os.path.isdir(f"{root}/{i:04d}")]
    llm = LLM(MERGED, enable_lora=True, max_lora_rank=16, max_model_len=2048, enforce_eager=True,
              gpu_memory_utilization=float(os.environ.get("GPU_UTIL", 0.45)))
    sp = SamplingParams(temperature=0, max_tokens=200)
    ids = {}
    def card(split, adapter):
        rows = [json.loads(l) for l in open(f"/workspace/data/burst/{split}.jsonl")]
        lora = LoRARequest(Path(adapter).name, ids.setdefault(adapter, len(ids) + 1), adapter) if adapter else None
        outs = [o.outputs[0].text for o in llm.chat([r["messages"] for r in rows], sp, lora_request=lora)]
        return scorecard(outs, [json.loads(r["ground_truth"]) for r in rows])
    val = {s: card("val", f"{root}/{s}") for s in steps}
    best = max(val, key=lambda s: (val[s]["All rules met %"], val[s]["Mean reward"]))
    res = {"val": val, "best": best, "best_path": f"{root}/{best}",
           "test": {"sft": card("test", None), f"sft+grpo (iteration {int(best)})": card("test", f"{root}/{best}")}}
    Path("/workspace/results").mkdir(exist_ok=True)
    json.dump(res, open("/workspace/results/burst-grpo.json", "w"), indent=1)
    print("RESULT", json.dumps(res["test"]), flush=True)

if __name__ == "__main__":
    {"merge": merge, "train": train, "evaluate": evaluate}[sys.argv[1]]()

### B3 · Start the GRPO Job

In [ ]:
# Create OpenShift AI resources with the workbench's own identity (no token to paste).
import yaml
from kubernetes import client as k8s, config, dynamic
config.load_incluster_config()
dyn = dynamic.DynamicClient(k8s.ApiClient())

def apply(manifest: str):
    """Create or update every object in a YAML string, in this project."""
    for obj in yaml.safe_load_all(manifest):
        res = dyn.resources.get(api_version=obj["apiVersion"], kind=obj["kind"])
        name = obj["metadata"]["name"]
        try:
            dyn.create(res, body=obj, namespace=NAMESPACE); print("created", obj["kind"], name)
        except k8s.ApiException as e:
            if e.status != 409: raise
            dyn.patch(res, body=obj, name=name, namespace=NAMESPACE, content_type="application/merge-patch+json")
            print("updated", obj["kind"], name)

def run_job(manifest: str, show=None):
    """(Re)start a Kubernetes Job and stream its log. `show` keeps only lines containing one of these words."""
    obj = yaml.safe_load(manifest); name = obj["metadata"]["name"]
    batch, core = k8s.BatchV1Api(), k8s.CoreV1Api()
    try:
        batch.delete_namespaced_job(name, NAMESPACE, propagation_policy="Foreground")
        while batch.list_namespaced_job(NAMESPACE, field_selector=f"metadata.name={name}").items: time.sleep(3)
    except k8s.ApiException: pass
    if NODE_SELECTOR: obj["spec"]["template"]["spec"]["nodeSelector"] = NODE_SELECTOR
    batch.create_namespaced_job(NAMESPACE, obj)
    while True:
        pods = core.list_namespaced_pod(NAMESPACE, label_selector=f"job-name={name}").items
        if pods and pods[0].status.phase in ("Running", "Succeeded", "Failed"): break
        time.sleep(5)
    for raw in core.read_namespaced_pod_log(pods[0].metadata.name, NAMESPACE, follow=True, _preload_content=False):
        for line in raw.decode(errors="replace").splitlines():
            if show is None or any(w in line for w in show): print(line, flush=True)
    s = batch.read_namespaced_job(name, NAMESPACE).status
    print(f"\nJob {name}: {'succeeded' if s.succeeded else 'FAILED - see the log above'}")

def wait_ready(kind, api_version, name, timeout=1800):
    res = dyn.resources.get(api_version=api_version, kind=kind); t0 = time.time()
    while time.time() - t0 < timeout:
        conds = (dyn.get(res, name=name, namespace=NAMESPACE).to_dict().get("status") or {}).get("conditions", [])
        if any(c["type"] == "Ready" and c["status"] == "True" for c in conds):
            return print(f"{name} is ready ({time.time() - t0:.0f} s)")
        time.sleep(10)
    raise TimeoutError(name)

In [ ]:
GRPO_JOB = f"""
apiVersion: batch/v1
kind: Job
metadata:
  name: sentinel7-grpo-burst
spec:
  backoffLimit: 0
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: grpo
        image: {TRAINING_IMAGE}
        command: ["/bin/bash", "-c"]
        args: ["set -e; cd /workspace/code; python grpo_burst.py merge; python grpo_burst.py train; python grpo_burst.py evaluate"]
        env:
        - {{name: ITERS, value: "24"}}
        - {{name: GPU_UTIL, value: "0.45"}}          # share of GPU memory for sampling; 0.45 of 48 GB in our run
        - {{name: MLFLOW_TRACKING_URI, value: "{os.environ.get('MLFLOW_TRACKING_URI', '')}"}}
        - {{name: MLFLOW_TRACKING_AUTH, value: kubernetes-namespaced}}
        - {{name: MLFLOW_EXPERIMENT_NAME, value: "{EXPERIMENT}"}}
        - {{name: HF_HOME, value: /workspace/.cache/huggingface}}
        - {{name: VLLM_USE_FLASHINFER_SAMPLER, value: "0"}}
        - {{name: PYTHONUNBUFFERED, value: "1"}}
        resources:
          requests: {{cpu: "2", memory: 24Gi, nvidia.com/gpu: "1"}}
          limits: {{cpu: "8", memory: 48Gi, nvidia.com/gpu: "1"}}
        volumeMounts: [{{name: work, mountPath: /workspace}}, {{name: dshm, mountPath: /dev/shm}}]
      volumes:
      - {{name: work, persistentVolumeClaim: {{claimName: {PVC}}}}}
      - {{name: dshm, emptyDir: {{medium: Memory, sizeLimit: 8Gi}}}}
"""
run_job(GRPO_JOB, show=["Iteration", "reward", "RESULT", "Error", "Traceback"])

In [ ]:
res = json.load(open(WS / "results/burst-grpo.json"))
print("best checkpoint on validation:", res["best"])
import pandas as pd
pd.DataFrame(res["test"]).T

**What we measured** (200 held-out reports, all seven rules met):

| Model | All rules met |
|---|---|
| Untuned Granite 4.1-3B | 42.5% |
| GRPO from the untuned model, no examples | 66.5% |
| SFT on the 173 checker-approved teacher answers | 88.0% |
| **That SFT + GRPO (24 iterations, best on validation)** | **95.0%** |

Missed items fell from 11% to 4%. We ran GRPO on a single dedicated 48 GB GPU with this same Red Hat training stack.
**Next:** notebook 04 serves the models.